minor project 2

Spend DNA

Name:Navyashree



- importing libraries
- read the transaction csv file

In [105]:
import pandas as pd
import numpy as np
df=pd.read_csv("rahul_transactions.csv")
print("The dataset loaded successfully")
print("Shape:",df.shape)


The dataset loaded successfully
Shape: (1328, 8)


In [106]:
print(df.head())

          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref  
0  678275.0   UPI  TXN190872  
1  681007.0   UPI  TXN143064  
2  484728.0  NEFT  TXN246316  
3 -748745.0   UPI  TXN569226  
4  680737.0   UPI  TXN968962  


In [107]:
print(df.dtypes)

Date            object
Time            object
Description     object
Type            object
Amount          object
Balance        float64
Mode            object
Ref             object
dtype: object


In [108]:
print(df.isnull().sum())

Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
dtype: int64


Feature 1: Transaction Parser
to clean raw transaction data and prepare

In [109]:
clean_data=df.copy()

In [110]:
# 1 Converting Date column into datetime format

clean_data['date'] = pd.to_datetime(
    clean_data['Date'],
    errors='coerce',
    dayfirst=True
)

In [111]:
2 #Clean the Amount column

clean_data['amount'] = (
    clean_data['Amount']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)


In [112]:
# 3 Standardizing transaction Type

# Converting Type values to lowercase
clean_data['type'] = clean_data['Type'].astype(str).str.lower()

clean_data['type'] = clean_data['type'].replace({
    'dr': 'debit',
    'debit': 'debit',
    'cr': 'credit',
    'credit': 'credit'
})

In [113]:
# 4 Remove duplicate transactions

# Count rows before removing duplicates
rows_before = len(clean_data)

# Removing exact duplicate rows
df_clean = clean_data.drop_duplicates()

duplicates_removed = rows_before - len(df_clean)

In [114]:
# 5 Check invalid dates and amounts

invalid_dates = df_clean['date'].isna().sum()
invalid_amounts = df_clean['amount'].isna().sum()

In [115]:
# 6 Displaying parser results

print("=" * 60)
print("TRANSACTION PARSER RESULTS")
print("=" * 60)

print(f"the Original transactions     : {rows_before}")
print(f"Duplicates removed       : {duplicates_removed}")
print(f"the Clean transactions        : {len(df_clean)}")
print(f"Invalid dates             : {invalid_dates}")
print(f"Invalid amounts           : {invalid_amounts}")

print("\nData types after cleaning:")
print(df_clean[['date', 'amount', 'type']].dtypes)

TRANSACTION PARSER RESULTS
the Original transactions     : 1328
Duplicates removed       : 18
the Clean transactions        : 1310
Invalid dates             : 1167
Invalid amounts           : 0

Data types after cleaning:
date      datetime64[ns]
amount            object
type              object
dtype: object


Feature 2- vendor extractor

In [116]:
#1 Inspect transaction descriptions

pd.set_option("display.max_colwidth", 150)

print("the Sample transaction descriptions:\n")
print(df["Description"].dropna().head(50).to_string(index=False))

the Sample transaction descriptions:

              AMAZON SELLER SVCS
                       BHIM-BMTC
NEFT-TECHCRUSH LABS-SALARY MAY24
            UPI-AMAN-8934@OKAXIS
                    BHIM-BLINKIT
                      BHIM ZEPTO
          UPI-UBER-2426@HDFCBANK
                    BHIM-BLINKIT
            POS SWIGGY BANGALORE
           UPI-GROWWPAY@HDFCBANK
                    OLA ELECTRIC
               BMS MOVIE TICKETS
                   POS OLA-PRIME
                SWIGGY-INSTAMART
              UPI-STARBUCKS@AXIS
            UPI-THIRDWAVE@OKAXIS
                ANI Technologies
                   BMTC BUS PASS
                    POS TRUFFLES
                  FLIPKART INDIA
           POS SWIGGY-RESTAURANT
               GROFERS INDIA P L
              POS UBER BANGALORE
           POS SWIGGY-RESTAURANT
           BANGALORE ELEC SUPPLY
                       TWC INDIA
        UPI-BESCOM-BILL@HDFCBANK
            UPI-AMAN-0816@OKAXIS
           ROPPEN TRANSPORTATION
     

In [117]:
# Create a column to store the standard vendor names
clean_data['vendor_name'] = 'Other'

In [118]:
# Dictionary containing vendor keywords and their common names
vendor_map = {
    'SWIGGY': 'Swiggy',
    'ZOMATO': 'Zomato',
    'AMAZON': 'Amazon',
    'BLINKIT': 'Blinkit',
    'ZEPTO': 'Zepto',
    'UBER': 'Uber',
    'OLA': 'Ola',
    'GROWW': 'Groww',
    'PRIME': 'Amazon Prime',
    'NETFLIX': 'Netflix',
    'SPOTIFY': 'Spotify',
    'YOUTUBE': 'YouTube',
    'BIGBASKET': 'BigBasket',
    'DMART': 'DMart',
    'JIO': 'Jio',
    'AIRTEL': 'Airtel'
}

# Match each keyword with the transaction description
for keyword, vendor in vendor_map.items():

    # Check whether the keyword appears in the description
    found = clean_data['Description'].astype(str).str.contains(
        keyword,
        case=False,
        na=False,
        regex=False
    )

    # Assigning the standard vendor name
    clean_data.loc[found, 'vendor_name'] = vendor

# Check for ATM transactions without using regular expressions
atm_found = clean_data['Description'].astype(str).str.contains(
    'ATM',
    case=False,
    na=False,
    regex=False
)

# Mark ATM transactions separately
clean_data.loc[atm_found, 'vendor_name'] = 'Cash Withdrawal'

# Display the vendor counts
print("the Vendor extraction completed!")
print("\n Vendor counts:")
print(clean_data['vendor_name'].value_counts())

# Count the total number of transactions
total_transactions = len(clean_data)

# Count the number of different vendors
unique_vendors = clean_data['vendor_name'].nunique()

# Checking whether vendor extraction was completed
vendor_extraction_successful = True

# Display vendor extraction results
print("VENDOR EXTRACTION SUMMARY")
print("=" * 40)
print("the Total Transactions       :", total_transactions)
print("Unique Vendors            :", unique_vendors)
print("Vendor Extraction Successful =", vendor_extraction_successful)

the Vendor extraction completed!

 Vendor counts:
vendor_name
Other              607
Swiggy             201
Zomato             124
Uber                71
Amazon              66
Zepto               60
Ola                 51
Blinkit             41
Amazon Prime        28
Cash Withdrawal     17
DMart               12
BigBasket           11
Netflix             10
Groww                9
Spotify              8
Jio                  6
Airtel               6
Name: count, dtype: int64
VENDOR EXTRACTION SUMMARY
the Total Transactions       : 1328
Unique Vendors            : 17
Vendor Extraction Successful = True


Feature 3: Category Tagger

In [119]:
# Create a new column for spending categories
clean_data['spend_category'] = 'Other'

In [120]:
# Assign categories using the vendor names
category_rules = {
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Blinkit': 'Quick Commerce',
    'Zepto': 'Quick Commerce',
    'Amazon': 'E-commerce',
    'Amazon Prime': 'Subscriptions',
    'Uber': 'Transport',
    'Ola': 'Transport',
    'BigBasket': 'Groceries',
    'DMart': 'Groceries',
    'Groww': 'Investments',
    'Netflix': 'Subscriptions',
    'Spotify': 'Subscriptions',
    'Jio': 'Utilities',
    'Airtel': 'Utilities',
    'Cash Withdrawal': 'Cash Withdrawal'
}

# Apply the category name for each matching vendor
for vendor, category in category_rules.items():

    # Find transactions belonging to the current vendor
    selected = clean_data['vendor_name'] == vendor

    # Add the required category
    clean_data.loc[selected, 'spend_category'] = category

# Check transaction descriptions for some additional categories
description_text = clean_data['Description'].astype(str)

# Mark cafe-related transactions
cafe_rows = description_text.str.contains(
    'CAFE',
    case=False,
    na=False,
    regex=False
)
clean_data.loc[cafe_rows, 'spend_category'] = 'Cafe'

# Mark restaurant-related transactions
restaurant_rows = description_text.str.contains(
    'RESTAURANT',
    case=False,
    na=False,
    regex=False
)
clean_data.loc[restaurant_rows, 'spend_category'] = 'Restaurants'

# Mark fuel-related transactions
fuel_rows = description_text.str.contains(
    'FUEL',
    case=False,
    na=False,
    regex=False
)
clean_data.loc[fuel_rows, 'spend_category'] = 'Fuel'

# Mark entertainment-related transactions
movie_rows = description_text.str.contains(
    'MOVIE',
    case=False,
    na=False,
    regex=False
)
clean_data.loc[movie_rows, 'spend_category'] = 'Entertainment'

# Mark investment-related transactions
investment_rows = description_text.str.contains(
    'INVEST',
    case=False,
    na=False,
    regex=False
)
clean_data.loc[investment_rows, 'spend_category'] = 'Investments'

# Display the number of transactions in each category
print("Category tagging completed!")
print("\nCategory counts:")
print(clean_data['spend_category'].value_counts())

Category tagging completed!

Category counts:
spend_category
Other              552
Food Delivery      300
Transport          122
Quick Commerce     101
Restaurants         67
E-commerce          66
Subscriptions       46
Groceries           23
Cash Withdrawal     17
Utilities           12
Investments          9
Entertainment        7
Cafe                 6
Name: count, dtype: int64


Feature 4 — Spending Overview

1 — Calculate credits and debits



In [124]:
# Make sure all transaction amounts are stored as numbers
clean_data['amount'] = pd.to_numeric(
    clean_data['amount'],
    errors='coerce'
)

In [125]:
#2 Total credits and debits
# Add all credit transactions
total_credits = clean_data.loc[
    clean_data['type'] == 'credit', 'amount'
].sum()

# Add all debit transactions
total_debits = clean_data.loc[
    clean_data['type'] == 'debit', 'amount'
].sum()

print("Total Credits :", round(total_credits, 2))
print("Total Debits  :", round(total_debits, 2))

#3 Net amount
# Find the difference between received and spent money
net_amount = total_credits - total_debits

print("Net Amount :", round(net_amount, 2))

#4 — Savings rate
# Calculate the savings percentage
if total_credits != 0:
    savings_rate = (net_amount / total_credits) * 100
else:
    savings_rate = 0

print("Savings Rate :", round(savings_rate, 2), "%")

#5 Find top spending category
# Keep only debit transactions
debit_data = clean_data[
    clean_data['type'] == 'debit'
]

# Add spending for each category
category_total = debit_data.groupby(
    'spend_category'
)['amount'].sum().sort_values(
    ascending=False
)

# Get the category with the highest spending
top_category = category_total.index[0]

print("Top Spending Category :", top_category)

#6 Feature 4 final summary
# Count all cleaned transactions
total_transactions = len(clean_data)

print("=" * 45)
print("FEATURE 4 - SPENDING OVERVIEW")
print("=" * 45)

print("Total Transactions :", total_transactions)
print("Total Credits      :", round(total_credits, 2))
print("Total Debits       :", round(total_debits, 2))
print("Net Amount         :", round(net_amount, 2))
print("Savings Rate       :", round(savings_rate, 2), "%")
print("Top Category       :", top_category)

#7 Category-wise spending
# Display the amount spent in each category
print("CATEGORY-WISE SPENDING")
print("=" * 45)
print(category_total)


Total Credits : 509774.0
Total Debits  : 1729708.0
Net Amount : -1219934.0
Savings Rate : -239.31 %
Top Spending Category : Other
FEATURE 4 - SPENDING OVERVIEW
Total Transactions : 1328
Total Credits      : 509774.0
Total Debits       : 1729708.0
Net Amount         : -1219934.0
Savings Rate       : -239.31 %
Top Category       : Other
CATEGORY-WISE SPENDING
spend_category
Other              970653.0
E-commerce         314962.0
Food Delivery      132249.0
Restaurants         69817.0
Quick Commerce      52826.0
Cash Withdrawal     45500.0
Transport           39070.0
Investments         38160.0
Groceries           33398.0
Subscriptions       21480.0
Utilities            6097.0
Entertainment        4081.0
Cafe                 1415.0
Name: amount, dtype: float64


Feature 5: Monthly Trend analysis

In [126]:
#1 Create month column
# Get the month from each transaction date
clean_data['month'] = clean_data['date'].dt.strftime('%Y-%m')

# Check the months available in the data
print("Months in the dataset:")
print(sorted(clean_data['month'].dropna().unique()))

Months in the dataset:
['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']


In [129]:
#2 Calculate monthly spending
# Select only debit transactions
monthly_debits = clean_data[
    clean_data['type'] == 'debit'
]

# Add spending for each month
monthly_spending = monthly_debits.groupby(
    'month'
)['amount'].sum()

# Display monthly spending
print("MONTHLY SPENDING")
print("=" * 40)
print(monthly_spending)

#3 Category-wise monthly spending
# Create a table showing category spending for each month
monthly_category = monthly_debits.pivot_table(
    index='spend_category',
    columns='month',
    values='amount',
    aggfunc='sum',
    fill_value=0
)

# Display the monthly category table
print("CATEGORY-WISE MONTHLY SPENDING")
print("=" * 50)
print(monthly_category)

#4 Find the highest spending month
# Find the month with the highest total spending
highest_month = monthly_spending.idxmax()
highest_amount = monthly_spending.max()

print("Highest Spending Month :", highest_month)
print("Amount Spent           :", round(highest_amount, 2))

#5 Find the lowest spending month
# Find the month with the lowest total spending
lowest_month = monthly_spending.idxmin()
lowest_amount = monthly_spending.min()

print("Lowest Spending Month :", lowest_month)
print("Amount Spent          :", round(lowest_amount, 2))

MONTHLY SPENDING
month
2024-01    14972.0
2024-02     6390.0
2024-03    26578.0
2024-04     5246.0
2024-05    21011.0
2024-06     9292.0
2024-07    23632.0
2024-08    41383.0
2024-09    14154.0
2024-10     5541.0
2024-11     6366.0
2024-12    31360.0
Name: amount, dtype: float64
CATEGORY-WISE MONTHLY SPENDING
month            2024-01  2024-02  2024-03  2024-04  2024-05  2024-06  \
spend_category                                                          
Cash Withdrawal      0.0      0.0      0.0      0.0      0.0      0.0   
E-commerce        2462.0      0.0      0.0      0.0      0.0      0.0   
Entertainment        0.0      0.0      0.0      0.0      0.0      0.0   
Food Delivery     1180.0   3454.0   1088.0    632.0   1159.0   1636.0   
Groceries            0.0      0.0   1213.0      0.0      0.0      0.0   
Investments          0.0      0.0      0.0      0.0      0.0      0.0   
Other             7885.0   1699.0  20316.0   4614.0  18166.0   7522.0   
Quick Commerce     825.0    697.

In [128]:
#6: Final Feature 5 summary
# Display the main results of monthly trend analysis
print("=" * 50)
print("FEATURE 5 - MONTHLY TREND ANALYSIS")
print("=" * 50)

print("Highest Spending Month :", highest_month)
print("Highest Spending       :", round(highest_amount, 2))
print("Lowest Spending Month  :", lowest_month)
print("Lowest Spending        :", round(lowest_amount, 2))

FEATURE 5 - MONTHLY TREND ANALYSIS
Highest Spending Month : 2024-08
Highest Spending       : 41383.0
Lowest Spending Month  : 2024-04
Lowest Spending        : 5246.0


Feature 6: Time-of-day patterns

In [131]:
#1 Extract transaction hour
# Take the hour part from the transaction time
clean_data['hour'] = clean_data['Time'].astype(str).str[:2]

# Convert the hour into numbers
clean_data['hour'] = pd.to_numeric(
    clean_data['hour'],
    errors='coerce'
)

# Check the extracted hours
print(clean_data['hour'].head())

0     3
1     5
2     9
3    14
4    14
Name: hour, dtype: int64


In [132]:
#2 Create time groups
# Create simple time periods based on the transaction hour
clean_data['time_period'] = 'Other'

clean_data.loc[
    clean_data['hour'].between(6, 11),
    'time_period'
] = 'Morning'

clean_data.loc[
    clean_data['hour'].between(12, 16),
    'time_period'
] = 'Afternoon'

clean_data.loc[
    clean_data['hour'].between(17, 20),
    'time_period'
] = 'Evening'

clean_data.loc[
    (clean_data['hour'] >= 21) | (clean_data['hour'] <= 5),
    'time_period'
] = 'Night'

print(clean_data['time_period'].value_counts())

#3-Category and hour table
# Select debit transactions for spending analysis
time_spending = clean_data[
    clean_data['type'] == 'debit'
]

# Create a table showing spending by category and hour
hour_category = time_spending.pivot_table(
    index='spend_category',
    columns='hour',
    values='amount',
    aggfunc='sum',
    fill_value=0
)

# Display the result
print("SPENDING BY CATEGORY AND HOUR")
print("=" * 50)
print(hour_category)
#4-Find busiest spending hour
# Add all debit amounts for each hour
hourly_total = time_spending.groupby(
    'hour'
)['amount'].sum().sort_values(
    ascending=False
)

# Get the hour with the highest spending
busiest_hour = hourly_total.index[0]
busiest_amount = hourly_total.iloc[0]

print("Busiest Spending Hour :", int(busiest_hour))
print("Amount Spent          :", round(busiest_amount, 2))


time_period
Evening      347
Night        339
Morning      323
Afternoon    319
Name: count, dtype: int64
SPENDING BY CATEGORY AND HOUR
hour                  0        1        2        3        4        5   \
spend_category                                                          
Cafe                 0.0      0.0      0.0      0.0      0.0      0.0   
Cash Withdrawal      0.0      0.0      0.0      0.0      0.0      0.0   
E-commerce       15584.0   6435.0   4304.0   8540.0    883.0   3587.0   
Entertainment        0.0    620.0      0.0      0.0      0.0      0.0   
Food Delivery      438.0   2837.0    928.0   1702.0   2243.0   3488.0   
Groceries            0.0   1536.0    635.0   7080.0   1169.0      0.0   
Investments       4883.0      0.0      0.0      0.0   3834.0   4496.0   
Other            15257.0  25792.0  12025.0  14513.0  29877.0  13061.0   
Quick Commerce     859.0   1402.0    697.0    732.0    768.0   1448.0   
Restaurants          0.0      0.0   1336.0      0.0    375.0 

In [133]:
#5-Final summary
# Display the main results of Feature 6
print("=" * 45)
print("FEATURE 6 - TIME-OF-DAY PATTERNS")
print("=" * 45)

print("Busiest Hour :", int(busiest_hour))
print("Amount Spent :", round(busiest_amount, 2))

print("\nSpending by Time Period:")
print(
    time_spending.groupby('time_period')['amount'].sum().sort_values(ascending=False))

FEATURE 6 - TIME-OF-DAY PATTERNS
Busiest Hour : 10
Amount Spent : 140571.0

Spending by Time Period:
time_period
Morning      468385.0
Afternoon    468252.0
Night        416272.0
Evening      376799.0
Name: amount, dtype: float64


Feature 7: Anomaly Detection


In [135]:
#1-Calculate category average
# Calculate the average transaction amount for each category
clean_data['category_mean'] = clean_data.groupby(
    'spend_category'
)['amount'].transform('mean')

# Calculate the standard deviation for each category
clean_data['category_std'] = clean_data.groupby(
    'spend_category'
)['amount'].transform('std')

In [136]:
#2-Calculate z-score
# Calculate how far each transaction is from its category average
clean_data['z_score'] = (
    (clean_data['amount'] - clean_data['category_mean'])
    / clean_data['category_std']
)

# Take only the positive z-score values for large transactions
clean_data['abs_z_score'] = clean_data['z_score'].abs()

print("Z-score calculation completed.")

#3-Find unusual transactions
# Select transactions that are more than 2 standard deviations away
unusual_transactions = clean_data[
    clean_data['abs_z_score'] > 2
].copy()

# Arrange the unusual transactions from highest to lowest z-score
unusual_transactions = unusual_transactions.sort_values(
    'abs_z_score',
    ascending=False
)

print("Total unusual transactions:", len(unusual_transactions))

#4-Display top 5 anomalies
# Select the five most unusual transactions
top_anomalies = unusual_transactions[
    ['date', 'Description', 'spend_category', 'amount', 'z_score']
].head(5)

# Display the results
print("TOP 5 ANOMALIES")
print("=" * 60)
print(top_anomalies.to_string(index=False))


Z-score calculation completed.
Total unusual transactions: 20
TOP 5 ANOMALIES
      date                      Description spend_category  amount  z_score
       NaT NEFT-TECHCRUSH LABS-SALARY MAY24          Other 85492.0 8.873775
2024-01-05 NEFT-TECHCRUSH LABS-SALARY MAY24          Other 85393.0 8.863167
       NaT NEFT-TECHCRUSH LABS-SALARY MAY24          Other 84736.0 8.792764
       NaT NEFT-TECHCRUSH LABS-SALARY MAY24          Other 84728.0 8.791906
       NaT NEFT-TECHCRUSH LABS-SALARY MAY24          Other 84724.0 8.791478


In [137]:
#5-Final summary
# Check whether any anomalies were detected
anomaly_found = len(unusual_transactions) > 0

print("=" * 45)
print("FEATURE 7 - ANOMALY DETECTION")
print("=" * 45)

print("Anomaly Detection Successful :", True)
print("Anomalies Found              :", anomaly_found)
print("Total Anomalies              :", len(unusual_transactions))

FEATURE 7 - ANOMALY DETECTION
Anomaly Detection Successful : True
Anomalies Found              : True
Total Anomalies              : 20


Feature 8: Spending archetype detection

In [138]:
#1-Calculate category percentages
# Calculate total debit spending
total_spending = clean_data.loc[
    clean_data['type'] == 'debit', 'amount'
].sum()

# Add spending amount for each category
category_amount = clean_data[
    clean_data['type'] == 'debit'
].groupby('spend_category')['amount'].sum()

# Convert category amounts into percentages
category_percent = (
    category_amount / total_spending
) * 100

# Display category percentages
print("CATEGORY SPENDING PERCENTAGE")
print("=" * 45)
print(category_percent.round(2))

CATEGORY SPENDING PERCENTAGE
spend_category
Cafe                0.08
Cash Withdrawal     2.63
E-commerce         18.21
Entertainment       0.24
Food Delivery       7.65
Groceries           1.93
Investments         2.21
Other              56.12
Quick Commerce      3.05
Restaurants         4.04
Subscriptions       1.24
Transport           2.26
Utilities           0.35
Name: amount, dtype: float64


In [142]:
#2 Check common spending patterns
# Get percentages for the main categories
food_delivery_pct = category_percent.get('Food Delivery', 0)
restaurant_pct = category_percent.get('Restaurants', 0)
cafe_pct = category_percent.get('Cafe', 0)
quick_pct = category_percent.get('Quick Commerce', 0)
ecommerce_pct = category_percent.get('E-commerce', 0)
investment_pct = category_percent.get('Investments', 0)
transport_pct = category_percent.get('Transport', 0)

# Combine food-related spending
food_total_pct = (
    food_delivery_pct +
    restaurant_pct +
    cafe_pct
)

print("the Food-related spending :", round(food_total_pct, 2), "%")
print("Quick Commerce        :", round(quick_pct, 2), "%")
print("E-commerce            :", round(ecommerce_pct, 2), "%")
print("Investments           :", round(investment_pct, 2), "%")
print("Transport             :", round(transport_pct, 2), "%")
#3 Create archetype list
# Store the spending personalities detected
archetypes = []

# Check Foodie pattern
if food_total_pct > 25:
    archetypes.append('The Foodie')

# Check Quick Commerce pattern
if quick_pct > 15:
    archetypes.append('Quick Commerce Junkie')

# Check Shopaholic pattern
if ecommerce_pct > 15:
    archetypes.append('Shopaholic')

# Check Investor pattern
if investment_pct > 15:
    archetypes.append('Investor')

# Check Cab Commuter pattern
if transport_pct > 10:
    archetypes.append('Cab Commuter')

print("Detected Archetypes:")
for personality in archetypes:
    print("-", personality)
#4 Check YOLO / Saver
# Check the overall savings behaviour
if savings_rate < 10:
    archetypes.append('YOLO Spender')

elif savings_rate > 40:
    archetypes.append('Disciplined Saver')

print("Updated Archetypes:")
for personality in archetypes:
    print("-", personality)
#5-Final result
# Remove any repeated archetype names
archetypes = list(set(archetypes))

the Food-related spending : 11.76 %
Quick Commerce        : 3.05 %
E-commerce            : 18.21 %
Investments           : 2.21 %
Transport             : 2.26 %
Detected Archetypes:
- Shopaholic
Updated Archetypes:
- Shopaholic
- YOLO Spender


In [141]:
# Display the final spending personality result
print("=" * 45)
print("FEATURE 8 - SPENDING ARCHETYPE DETECTION")
print("=" * 45)

print("Total Archetypes Found :", len(archetypes))

for personality in archetypes:
    print("-", personality)

FEATURE 8 - SPENDING ARCHETYPE DETECTION
Total Archetypes Found : 2
- YOLO Spender
- Shopaholic


Final Spend DNA report

In [146]:
#1 Report heading
# Print the report heading
print("=" * 65)
print("                 SPEND DNA REPORT")
print("                 6 MONTHS - JAN TO JUN 2024")
print("=" * 65)

#2 Executive Summary
# Print the main financial details
print("\n EXECUTIVE SUMMARY")
print("*" * 45)

print("the Total credits   :", round(total_credits, 2))
print("Total debits        :", round(total_debits, 2))
print("Net change          :", round(net_amount, 2))
print("Savings rate        :", round(savings_rate, 2), "%")
print("Transactions        :", len(clean_data))
print("Unique vendors      :", clean_data['vendor_name'].nunique())

#3 Top Categories
# Calculate category amount and percentage
category_report = category_amount.sort_values(
    ascending=False
)

category_share = (
    category_report / total_debits
) * 100

# Print the top five spending categories
print("\n TOP CATEGORIES (% OF DEBIT TOTAL)")
print("-" * 55)

for category in category_report.head(5).index:
    amount = category_report[category]
    percent = category_share[category]

    print(
        f"{category:<20} {percent:>6.1f}%   Rs. {amount:>10.2f}"
    )

#4 Top Vendors
# Calculate spending for each vendor
vendor_report = debit_data.groupby(
    'vendor_name'
)['amount'].sum().sort_values(
    ascending=False
)

# Count transactions for each vendor
vendor_orders = debit_data['vendor_name'].value_counts()

# Print the top five vendors
print("\nTOP VENDORS")
print("*" * 55)

for vendor in vendor_report.head(5).index:
    amount = vendor_report[vendor]
    orders = vendor_orders.get(vendor, 0)

    print(
        f"{vendor:<20} Rs. {amount:>10.2f}   ({orders} orders)"
    )
#5 Time-of-Day Patterns
# Find Food Delivery transactions
food_data = clean_data[
    clean_data['spend_category'] == 'Food Delivery'
]

# Count food orders made after 9 PM
late_food = food_data[
    (food_data['hour'] >= 21) |
    (food_data['hour'] <= 1)
]

# Calculate the percentage
if len(food_data) > 0:
    late_food_percent = (
        len(late_food) / len(food_data)
    ) * 100
else:
    late_food_percent = 0

# Find the busiest food delivery hour
food_hour_count = food_data['hour'].value_counts()

if len(food_hour_count) > 0:
    food_peak_hour = food_hour_count.index[0]
else:
    food_peak_hour = "N/A"

print("\nTIME-OF-DAY PATTERNS")
print("*" * 55)

print(
    "Food Delivery peak hour :",
    food_peak_hour
)

print(
    "Food orders after 9 PM  :",
    round(late_food_percent, 2), "%"
)
#6 Monthly Food Delivery Trend
# Select Food Delivery transactions
food_monthly = food_data.groupby(
    'month'
)['amount'].sum()

# Print monthly Food Delivery spending
print("\nMONTHLY TREND (FOOD DELIVERY)")
print("*" * 54)

for month, amount in food_monthly.items():
    print(
        f"{month:<10} Rs. {amount:>10.2f}"
    )
#7 Top Anomalies
# Print the five largest unusual transactions
print("\nTOP ANOMALIES")
print("*" * 65)

for _, row in top_anomalies.iterrows():
    print(
        f"{row['date']} - "
        f"{row['Description']:<25} "
        f"Rs. {row['amount']:>10.2f} "
        f"(z={row['z_score']:.2f})"
    )

#8 Spending Archetypes
# Print the spending personalities detected
print("\nSPENDING ARCHETYPES")
print("*" * 55)

if len(archetypes) == 0:
    print("No spending archetype matched the given rules.")
else:
    for personality in archetypes:
        print("->", personality)

#9 Key Insights
# Find the category with the highest share
main_category = category_share.idxmax()
main_category_percent = category_share.max()

# Find the vendor with the highest spending
main_vendor = vendor_report.index[0]

print("\nKEY INSIGHTS")
print("*" * 55)

print(
    "1. The highest spending category was",
    main_category,
    "with",
    round(main_category_percent, 2),
    "% of debit spending."
)

print(
    "2. The highest-spending vendor was",
    main_vendor,
    "with Rs.",
    round(vendor_report.iloc[0], 2),
    "spent."
)

print(
    "3. Food Delivery spending after 9 PM was",
    round(late_food_percent, 2),
    "% of Food Delivery transactions."
)

print(
    "4. Total unusual transactions detected:",
    len(unusual_transactions)
)

#10 Final closing line
# Finish the printed report
print("\n" + "=" * 65)
print("                    END OF REPORT")
print("=" * 65)

                 SPEND DNA REPORT
                 6 MONTHS - JAN TO JUN 2024

 EXECUTIVE SUMMARY
*********************************************
the Total credits   : 509774.0
Total debits        : 1729708.0
Net change          : -1219934.0
Savings rate        : -239.31 %
Transactions        : 1328
Unique vendors      : 17

 TOP CATEGORIES (% OF DEBIT TOTAL)
-------------------------------------------------------
Other                  56.1%   Rs.  970653.00
E-commerce             18.2%   Rs.  314962.00
Food Delivery           7.6%   Rs.  132249.00
Restaurants             4.0%   Rs.   69817.00
Quick Commerce          3.1%   Rs.   52826.00

TOP VENDORS
*******************************************************
Other                Rs. 1035047.00   (601 orders)
Amazon               Rs.  314962.00   (66 orders)
Swiggy               Rs.   86114.00   (201 orders)
Zomato               Rs.   57054.00   (124 orders)
Cash Withdrawal      Rs.   45500.00   (17 orders)

TIME-OF-DAY PATTERNS
**********